# 05 Survival: Kaplan-Meier, the log-rank test and the Cox model

simplydata's survival words, checked on the **lung** data (Loprinzi et al. 1994, North Central Cancer Treatment
Group): survival in days of 228 patients with advanced lung cancer, shipped with R's survival package (Therneau).
simplydata computes these as R's survival package does, so R survival is the original.

**Only simplydata runs in this notebook.** The reference values are in `references/05_survival.csv`, written by
`scripts/references/05_survival.py`. The survival package's manual prints no numbers for these models on lung, so R
is run once there. Kaplan-Meier and the log-rank test are closed formulas: they must agree to a relative difference
of 1e-10. The Cox model is fitted by iteration: 1e-6 (R run to eps 1e-12). Each check prints its source.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_row, read_rows, read_rows_by, versions

NOTEBOOK = "05_survival"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)
lung = quiet(sdata.table, data_file(NOTEBOOK, "lung"), separator=",")
lung.show_first_rows()

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


,inst,time,status,age,sex,ph.ecog,ph.karno,pat.karno,meal.cal,wt.loss,died
1,3.0,306,2,74,male,1.0,90.0,100.0,1175.0,NaN,1
2,3.0,455,2,68,male,0.0,90.0,90.0,1225.0,15.0,1
3,3.0,1010,1,56,male,0.0,90.0,90.0,NaN,15.0,0
4,5.0,210,2,57,male,1.0,90.0,60.0,1150.0,11.0,1
5,1.0,883,2,60,male,0.0,100.0,90.0,NaN,0.0,1


## Kaplan-Meier: `table.find_survival_over_time`

The share still alive over time, for women and men, with 95 % confidence intervals on the log scale (R's default,
Greenwood's variance). Compared with R `summary(survfit(Surv(time, died) ~ 1))` for each sex at every event time:
the number of event times, then each column (time, survival, its interval, at risk, events) by its largest relative
difference.

In [2]:
km = lung.find_survival_over_time(time="time", event="died", for_each_group_in="sex")

The event is "died" = 1 (165 of 228 rows); the other rows count as followed without the event until their "time" (censored).
Median time until the event (by then half have had it), with its 95 % CI:
  female: median "time" 426 (95 % CI 348 to 550; 90 rows, 53 events).
  male: median "time" 270 (95 % CI 212 to 310; 138 rows, 112 events).
Method: Kaplan-Meier estimate with 95 % confidence intervals on the log scale (Greenwood), as R survfit(Surv(time, died) ~ sex).


In [3]:
rows = read_rows(km)
for sex in ("female", "male"):
    group = [row for row in rows if row["sex"] == sex]
    dataset = f"lung ({sex})"
    checks.check(dataset, "find_survival_over_time", "event times", len(group))
    for column in ("time", "survival", "95 % CI low", "95 % CI high", "at risk", "events"):
        checks.check(dataset, "find_survival_over_time", f"{column} at every event time", [row[column] for row in group])

agrees  lung (female) | find_survival_over_time | event times: simplydata 51, R summary(survfit) 51; difference 0.00e+00 (allowed 0e+00)
      Source: R summary(survfit) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/summary.survfit.html (version 3.8.12)
agrees  lung (female) | find_survival_over_time | time at every event time: largest relative difference over 51 values: simplydata 0, R summary(survfit) 0; difference 0.00e+00 (allowed 1e-10)
      Source: R summary(survfit) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/summary.survfit.html (version 3.8.12)
agrees  lung (female) | find_survival_over_time | survival at every event time: largest relative difference over 51 values: simplydata 4.52809572503e-16, R summary(survfit) 0; difference 4.53e-16 (allowed 1e-10)
      Source: R summary(survfit) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/summary.survfit.html (version 3.8.12)
agrees  lung (female) | 

### The median time

`find_survival_over_time(..., show="median")` gives back the median time and its 95 % interval for each group as a
table (columns "median", "95 % CI low" and "95 % CI high"; an empty cell means not reached), as R
`print(survfit(...))` shows them. Compared with R `summary(survfit(...))$table` (relative difference 1e-10).

In [4]:
medians = read_rows_by(lung.find_survival_over_time(time="time", event="died", for_each_group_in="sex", show="median"), "sex")
for sex in ("female", "male"):
    checks.check_values(f"lung ({sex})", "find_survival_over_time (median)",
                        {"median": medians[sex]["median"], "median 95 % CI low": medians[sex]["95 % CI low"],
                         "median 95 % CI high": medians[sex]["95 % CI high"]})

The event is "died" = 1 (165 of 228 rows); the other rows count as followed without the event until their "time" (censored).
Median time until the event (by then half have had it), with its 95 % CI:
  female: median "time" 426 (95 % CI 348 to 550; 90 rows, 53 events).
  male: median "time" 270 (95 % CI 212 to 310; 138 rows, 112 events).
Method: Kaplan-Meier estimate with 95 % confidence intervals on the log scale (Greenwood), as R survfit(Surv(time, died) ~ sex).
Gives back the median times as a table (show="median"); an empty cell means not reached.
agrees  lung (female) | find_survival_over_time (median) | median: simplydata 426, R survfit 426; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survfit (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/print.survfit.html (version 3.8.12)
agrees  lung (female) | find_survival_over_time (median) | median 95 % CI low: simplydata 348, R survfit 348; relative difference 0.00e+00 (allowed 1e-10)
      Sourc

## The log-rank test: `table.compare_survival`

Do women and men differ in survival? Compared with R `survdiff` (the chi-square, its p-value, and the expected deaths
in each group).

In [5]:
result = lung.compare_survival(time="time", event="died", for_each_group_in="sex")

The event is "died" = 1 (165 of 228 rows); the other rows count as followed without the event until their "time" (censored).
Events in each "sex" group, observed vs expected if all groups were alike: female 53 vs 73.4 and male 112 vs 91.6.
If the groups truly had the same time until the event, differences this big or bigger would turn up in about 1 of 1000 studies like this one (p = 0.001).
Test: log-rank test (chi-square = 10.3, df = 1), as R survdiff(Surv(time, died) ~ sex); it compares the whole curves, giving every event time the same weight.


In [6]:
row = read_row(result)
checks.check_values("lung", "compare_survival", {q: row[q] for q in ["statistic", "p-value", "expected events in female",
                                                                      "expected events in male"]})

agrees  lung | compare_survival | statistic: simplydata 10.3267419549, R survdiff 10.3267419549; relative difference 1.03e-15 (allowed 1e-10)
      Source: R survdiff (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survdiff.html (version 3.8.12)
agrees  lung | compare_survival | p-value: simplydata 0.00131116452036, R survdiff 0.00131116452036; relative difference 6.40e-14 (allowed 1e-10)
      Source: R survdiff (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survdiff.html (version 3.8.12)
agrees  lung | compare_survival | expected events in female: simplydata 73.4182609704, R survdiff 73.4182609704; relative difference 1.94e-16 (allowed 1e-10)
      Source: R survdiff (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survdiff.html (version 3.8.12)
agrees  lung | compare_survival | expected events in male: simplydata 91.5817390296, R survdiff 91.5817390296; relative difference 4.66e-16 (allowed 1e-10)
      So

4

## The Cox model: `table.explain_survival`

Hazard ratios for age, sex (male against female) and ECOG performance score, with Efron's method for tied times. One
row has no ECOG score and is left out. Compared with R `coxph()` run to tight convergence (eps 1e-12; its default
stops sooner): each coefficient (log hazard ratio), its standard error and p-value; and the proportional-hazards check
against R `cox.zph()`.

In [7]:
cox = lung.explain_survival(time="time", event="died", using=["age", "sex", "ph.ecog"])

Note: 1 row with an empty cell in "ph.ecog" was skipped.


The event is "died" = 1 (164 of 227 rows); the other rows count as followed without the event until their "time" (censored).
Hazard ratios from 227 rows with 164 events (above 1: a higher risk of the event at any moment; below 1: a lower risk):
  each 1 more "age": hazard ratio 1.01 (95 % CI 0.993 to 1.03); about 23 of 100 studies would show this much if there were truly none (p = 0.232).
  "sex" male vs female: hazard ratio 1.74 (95 % CI 1.25 to 2.41); fewer than 1 in 1000 studies would show this much if there were truly none (p = 0.0010).
  each 1 more "ph.ecog": hazard ratio 1.59 (95 % CI 1.27 to 1.99); fewer than 1 in 1000 studies would show this much if there were truly none (p < 0.0001).
The model as a whole (likelihood-ratio test, chi-square = 30.5, df = 3): fewer than 1 in 1000 studies would show this much if none of the columns truly mattered (p < 0.0001).
Harrell's C = 0.637 (standard error 0.0251): in 63.7 % of the pairs where one row had the event first, the model gave that

In [8]:
found = read_rows_by(cox, "term")
values = {}
for term in ("age", "sex: male vs female", "ph.ecog"):
    for quantity in ("coefficient", "standard error", "p-value", "proportional hazards check p-value"):
        values[f"{term} {quantity}"] = found[term][quantity]
checks.check_values("lung", "explain_survival", values)

agrees  lung | explain_survival | age coefficient: simplydata 0.0110667645961, R coxph 0.0110667645961; relative difference 3.92e-15 (allowed 1e-06)
      Source: R coxph (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/coxph.html (version 3.8.12)
agrees  lung | explain_survival | age standard error: simplydata 0.00926741101437, R coxph 0.00926741101437; relative difference 1.09e-14 (allowed 1e-06)
      Source: R coxph (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/coxph.html (version 3.8.12)
agrees  lung | explain_survival | age p-value: simplydata 0.232415679512, R coxph 0.232415679512; relative difference 4.78e-16 (allowed 1e-06)
      Source: R coxph (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/coxph.html (version 3.8.12)
agrees  lung | explain_survival | age proportional hazards check p-value: simplydata 0.664596763116, R cox.zph 0.664596763193; relative difference 1.16e-10 (allowed 1e-06)
      Sour

12

## Result

In [9]:
checks.finish()

05_survival: 36 comparisons - 36 agrees
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,05_survival,lung (female),find_survival_over_time,event times,5.100000e+01,51.000000,R summary(survfit),0.000000e+00,0.000000e+00,absolute,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1,05_survival,lung (female),find_survival_over_time,time at every event time: largest relative dif...,0.000000e+00,0.000000,R summary(survfit),0.000000e+00,1.000000e-10,absolute,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2,05_survival,lung (female),find_survival_over_time,survival at every event time: largest relative...,4.528096e-16,0.000000,R summary(survfit),4.528096e-16,1.000000e-10,absolute,4.528096e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3,05_survival,lung (female),find_survival_over_time,95 % CI low at every event time: largest relat...,4.516429e-16,0.000000,R summary(survfit),4.516429e-16,1.000000e-10,absolute,4.516429e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
4,05_survival,lung (female),find_survival_over_time,95 % CI high at every event time: largest rela...,1.029215e-15,0.000000,R summary(survfit),1.029215e-15,1.000000e-10,absolute,1.029215e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
5,05_survival,lung (female),find_survival_over_time,at risk at every event time: largest relative ...,0.000000e+00,0.000000,R summary(survfit),0.000000e+00,1.000000e-10,absolute,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
6,05_survival,lung (female),find_survival_over_time,events at every event time: largest relative d...,0.000000e+00,0.000000,R summary(survfit),0.000000e+00,1.000000e-10,absolute,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
7,05_survival,lung (male),find_survival_over_time,event times,9.900000e+01,99.000000,R summary(survfit),0.000000e+00,0.000000e+00,absolute,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
8,05_survival,lung (male),find_survival_over_time,time at every event time: largest relative dif...,0.000000e+00,0.000000,R summary(survfit),0.000000e+00,1.000000e-10,absolute,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
9,05_survival,lung (male),find_survival_over_time,survival at every event time: largest relative...,1.243464e-15,0.000000,R summary(survfit),1.243464e-15,1.000000e-10,absolute,1.243464e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
